# Defend the center: controlled LIF hyperparameter sweeps

Ten unique configurations cover five one-factor-at-a-time categories: learning rate,
frame skip, frame stacking, ammo input and ammo penalty. A shared baseline is reused
in each category. This is a screening experiment, not a full Cartesian grid.

**Training is disabled until ENABLE_TRAINING=True.** Start with smoke mode and one
training seed. Full defaults use milestones at 2M/4M/8M Sample Factory steps and 300
matching evaluation episodes per milestone. Confirm finalists with seeds 0, 1, 2.
Ten configurations at 8M imply 80M requested steps per training seed. Three seeds
imply 30 fresh runs and 240M steps. More observations, checkpoints and evaluation
increase runtime substantially; GPU memory must be available.

No original notebook or existing checkpoint is modified. Each job uses a fresh process
and a separate sweep directory. This notebook's controller does not initialize CUDA.
Upload `sweep_support.py` and `sweep_worker.py` alongside this notebook on the server.


In [ ]:
import os, sys, json, hashlib, itertools, subprocess, time, signal
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
ROOT = next((p for p in [Path.cwd(),*Path.cwd().parents]
             if (p/"defend the center"/"sweep_worker.py").is_file()),None)
if ROOT is None: raise RuntimeError("Put the notebook and both sweep helper files in defend the center")
WORK_DIR = ROOT/"defend the center"
OUTPUT_ROOT = Path(os.environ.get("DEFEND_OUTPUT_ROOT",str(WORK_DIR/"outputs"))).expanduser().resolve()
ENABLE_TRAINING = False
SMOKE_TEST = True
TRAINING_SEEDS = [0]            # screening; finalists should also run seeds 1 and 2
SWEEP_TAG = "lif_screen_v1"
CATEGORIES = ["learning_rate","frame_skip","stack","ammo","penalty"]
# Reduce category list to limit cost; baseline always included.
MILESTONES = [100_000] if SMOKE_TEST else [2_000_000,4_000_000,8_000_000]
EVAL_SEEDS = list(range(10000,10010 if SMOKE_TEST else 10300))
CALIBRATION_FRAMES = 64 if SMOKE_TEST else 800
ACTIVITY_FRAMES = 20 if SMOKE_TEST else 500
CPU_BUDGET = int(os.environ.get("SLURM_CPUS_PER_TASK",len(os.sched_getaffinity(0))))
WORKERS = max(1,min(8,CPU_BUDGET,int(os.environ.get("DOOM_NUM_WORKERS",max(1,CPU_BUDGET-2)))))
SWEEP_ROOT = OUTPUT_ROOT/"sweeps"/SWEEP_TAG
BASELINE = dict(learning_rate=1e-4,frame_skip=4,stack=1,ammo=False,penalty=0.)
VALUES = dict(learning_rate=[3e-5,1e-4,3e-4],frame_skip=[4,2],stack=[1,2,4],
              ammo=[False,True],penalty=[0.,.01,.03,.1])
assert set(CATEGORIES)<=set(VALUES)
assert TRAINING_SEEDS and EVAL_SEEDS and MILESTONES==sorted(set(MILESTONES))


## Build and review the plan

All candidates use beta=.9, T=8, batch=1024, 72x128 images and no recurrent core.
Stacking concatenates RGB images along channels (3, 6 or 12 channels), with repeated
initial frames at reset. Ammo is a bounded scalar (ammo/26) appended to the 512 features;
it is intentionally additional information, so compare a matched ANN before claiming
an ANN-vs-SNN advantage. Penalty values subtract this amount per ammunition unit used
in training; evaluation always restores the original reward.

Frame skip is a protocol change, not just a learning parameter. Requested steps are
Sample Factory's reported training counter. Record actual counters, wall time and game
seconds; do not assume identical decision counts or compute budgets across skips.
Changing stack/skip also changes calibration observations; it is not a pure neuron ablation.


In [ ]:
def config_id(config):
    return hashlib.sha256(json.dumps(config,sort_keys=True).encode()).hexdigest()[:10]
candidates={config_id(BASELINE):dict(config=BASELINE.copy(),categories=["baseline"])}
for category in CATEGORIES:
    for value in VALUES[category]:
        cfg=dict(BASELINE,**{category:value});key=config_id(cfg)
        if key not in candidates:candidates[key]=dict(config=cfg,categories=[])
        candidates[key]["categories"].append(category)
JOBS=[]
for key,item in candidates.items():
    for seed in TRAINING_SEEDS:
        JOBS.append(dict(name=f"lif_{key}_seed{seed}",seed=seed,config=item["config"],
            category=item["categories"],root=str(SWEEP_ROOT),workers=WORKERS,
            milestones=MILESTONES,eval_seeds=EVAL_SEEDS,
            calibration_frames=CALIBRATION_FRAMES,activity_frames=ACTIVITY_FRAMES))
print("Configurations:",len(candidates),"fresh runs:",len(JOBS))
print("Requested total training steps:",len(JOBS)*MILESTONES[-1])
print("Evaluation episodes:",len(JOBS)*len(MILESTONES)*len(EVAL_SEEDS))
for job in JOBS:print(job["name"],job["category"],job["config"])


## Launch sequential jobs (explicit opt-in)

Completed jobs are reused only when their submitted configuration matches exactly.
Failed jobs stop the sweep and preserve logs. Incomplete training is not silently resumed:
inspect the log and use a new sweep tag for a clean restart. Changing settings also needs
a new tag. Each candidate trains in stages with latest-checkpoint evaluation between
stages; every candidate uses this same schedule. Training wall time excludes calibration
and evaluation. On interruption the controller terminates only its own process group.


In [ ]:
MODULE_SOURCE = '"""Directly trained spiking policy for Sample Factory (ViZDoom doom_defend_the_center).\n\nSame layer sizes as the ANN baseline (convnet_simple): 3 convs + a 512 layer, but every\nhidden layer is an leaky integrate-and-fire (LIF) layer unrolled over T time steps and trained\nwith surrogate gradients. State is reset for every frame (no recurrence across frames).\n"""\nimport numpy as np\nimport torch\nfrom torch import nn\n\nfrom sample_factory.algo.utils.torch_utils import calc_num_elements\nfrom sample_factory.model.encoder import Encoder\n\n\n# ----------------------------------------------------------------------------------------\n# Neuron: Heaviside spike in the forward pass, triangular surrogate in the backward pass\n# ----------------------------------------------------------------------------------------\nclass SpikeFn(torch.autograd.Function):\n    @staticmethod\n    def forward(ctx, x):  # x = membrane - threshold\n        ctx.save_for_backward(x)\n        return (x >= 0).to(x.dtype)\n\n    @staticmethod\n    def backward(ctx, grad_out):\n        (x,) = ctx.saved_tensors\n        return grad_out * torch.clamp(1.0 - x.abs(), min=0.0)  # triangular, width 1\n\n\ndef lif_step(v, current, threshold=1.0, v_init=0.5, detach_reset=True, beta=0.9):\n    """One LIF step: decay voltage, add current, spike, subtract threshold.\n\n    beta=1 recovers the IF update; current is not multiplied by (1-beta)."""\n    if v is None:\n        v = torch.full_like(current, v_init * threshold)\n    v = beta * v + current\n    s = SpikeFn.apply(v - threshold)\n    v = v - (s.detach() if detach_reset else s) * threshold\n    return v, s\n\n\n# ----------------------------------------------------------------------------------------\n# The spiking trunk (no Sample Factory dependency in the maths, so it can be evaluated alone)\n# ----------------------------------------------------------------------------------------\nclass SpikingTrunk(nn.Module):\n    def __init__(self, in_channels=3, obs_hw=(72, 128), T=8, v_init=0.5,\n                 gains=(1.0, 1.0, 1.0, 1.0), feat_scale=4.0, beta=0.9):\n        super().__init__()\n        if not 0.0 <= float(beta) <= 1.0:\n            raise ValueError("beta must be in [0, 1]")\n        self.register_buffer("beta", torch.tensor(float(beta)))\n        self.T = int(T)\n        self.v_init = float(v_init)\n        self.conv1 = nn.Conv2d(in_channels, 32, 8, stride=4)\n        self.conv2 = nn.Conv2d(32, 64, 4, stride=2)\n        self.conv3 = nn.Conv2d(64, 128, 3, stride=2)\n        # local (unregistered) stack, only used to compute the flattened size; keeps state_dict keys unique\n        self.flat_size = calc_num_elements(nn.Sequential(self.conv1, self.conv2, self.conv3), (in_channels, *obs_hw))\n        self.fc = nn.Linear(self.flat_size, 512)\n        # fixed (non-trainable) per-layer current gains and output scale: buffers, so they are saved\n        # in the checkpoint and are not touched by Sample Factory\'s weight initialisation\n        self.register_buffer("gains", torch.tensor(list(gains), dtype=torch.float32))\n        self.register_buffer("feat_scale", torch.tensor(float(feat_scale)))\n        self.out_size = 512\n\n    def forward(self, x, T=None, return_spikes=False):\n        T = self.T if T is None else int(T)\n        g = self.gains\n        c1 = self.conv1(x) * g[0]  # direct encoding: identical every step, computed once\n        v1 = v2 = v3 = v4 = None\n        acc = 0\n        spikes = [0.0, 0.0, 0.0, 0.0]\n        for _ in range(T):\n            v1, s1 = lif_step(v1, c1, v_init=self.v_init, beta=self.beta)\n            v2, s2 = lif_step(v2, self.conv2(s1) * g[1], v_init=self.v_init, beta=self.beta)\n            v3, s3 = lif_step(v3, self.conv3(s2) * g[2], v_init=self.v_init, beta=self.beta)\n            v4, s4 = lif_step(v4, self.fc(s3.flatten(1)) * g[3], v_init=self.v_init, beta=self.beta)\n            acc = acc + s4\n            if return_spikes:\n                for k, s in enumerate((s1, s2, s3, s4)):\n                    spikes[k] = spikes[k] + s.detach().flatten(1).sum(1)  # per-sample spike count\n        feats = acc * (self.feat_scale / T)  # mean firing rate of the 512 layer, scaled\n        if return_spikes:\n            return feats, torch.stack(spikes, 1)  # (B, 4) spikes per decision per layer\n        return feats\n\n\n# ----------------------------------------------------------------------------------------\n# Sample Factory glue\n# ----------------------------------------------------------------------------------------\ndef add_snn_args(parser):\n    p = parser\n    p.add_argument("--snn_beta", default=0.9, type=float, help="Fixed LIF membrane retention in [0, 1]")\n    p.add_argument("--snn_T", default=8, type=int, help="Time steps per frame for the spiking encoder")\n    p.add_argument("--snn_v_init", default=0.5, type=float, help="Initial membrane potential / threshold")\n    p.add_argument("--snn_gains", default="1,1,1,1", type=str,\n                   help="Comma separated fixed current gains for the 4 spiking layers (from calibration)")\n    p.add_argument("--snn_feat_scale", default=4.0, type=float, help="Scale on the mean firing rate fed to the heads")\n\n\nclass SpikingEncoder(Encoder):\n    def __init__(self, cfg, obs_space):\n        super().__init__(cfg)\n        obs_shape = obs_space["obs"].shape\n        gains = [float(v) for v in str(cfg.snn_gains).split(",")]\n        assert len(gains) == 4, "--snn_gains needs 4 comma separated numbers"\n        self.trunk = SpikingTrunk(obs_shape[0], tuple(obs_shape[1:]), cfg.snn_T, cfg.snn_v_init,\n                                  gains, cfg.snn_feat_scale, beta=cfg.snn_beta)\n        self.encoder_out_size = self.trunk.out_size\n\n    def forward(self, obs_dict):\n        return self.trunk(obs_dict["obs"])\n\n    def get_out_size(self):\n        return self.encoder_out_size\n\n\ndef make_snn_encoder(cfg, obs_space):\n    return SpikingEncoder(cfg, obs_space)\n\n\n# ----------------------------------------------------------------------------------------\n# Data-dependent gain calibration (so the network does not start silent)\n# ----------------------------------------------------------------------------------------\n@torch.no_grad()\ndef calibrate_gains(frames, T=8, target_rate=0.15, v_init=0.5, iters=18, device="cpu", seed=0, beta=0.9):\n    """Pick fixed per-layer current gains so each layer\'s mean firing rate is ~target_rate at\n    initialisation, using real (already standardised) frames. Weights use the same init Sample\n    Factory applies (orthogonal, gain 1, zero bias)."""\n    torch.manual_seed(seed)\n    trunk = SpikingTrunk(frames.shape[1], tuple(frames.shape[2:]), T, v_init, beta=beta).to(device)\n    for m in (trunk.conv1, trunk.conv2, trunk.conv3, trunk.fc):\n        nn.init.orthogonal_(m.weight, gain=1.0)\n        nn.init.zeros_(m.bias)\n    x = frames.to(device)\n    gains = [1.0, 1.0, 1.0, 1.0]\n\n    def rate_of_layer(k, g):\n        gg = list(gains)\n        gg[k] = g\n        c1 = trunk.conv1(x) * gg[0]\n        v = [None] * 4\n        total, count = 0.0, 0\n        for _ in range(T):\n            v[0], s1 = lif_step(v[0], c1, v_init=v_init, beta=beta)\n            if k == 0:\n                total += s1.sum().item(); count += s1.numel(); continue\n            v[1], s2 = lif_step(v[1], trunk.conv2(s1) * gg[1], v_init=v_init, beta=beta)\n            if k == 1:\n                total += s2.sum().item(); count += s2.numel(); continue\n            v[2], s3 = lif_step(v[2], trunk.conv3(s2) * gg[2], v_init=v_init, beta=beta)\n            if k == 2:\n                total += s3.sum().item(); count += s3.numel(); continue\n            v[3], s4 = lif_step(v[3], trunk.fc(s3.flatten(1)) * gg[3], v_init=v_init, beta=beta)\n            total += s4.sum().item(); count += s4.numel()\n        return total / count\n\n    for k in range(4):\n        lo, hi = -4.0, 4.0  # search log10(gain)\n        for _ in range(iters):\n            mid = 0.5 * (lo + hi)\n            if rate_of_layer(k, 10 ** mid) < target_rate:\n                lo = mid\n            else:\n                hi = mid\n        gains[k] = float(10 ** (0.5 * (lo + hi)))\n    rates = [rate_of_layer(k, gains[k]) for k in range(4)]\n    return gains, rates\n\n\n# ----------------------------------------------------------------------------------------\n# Standalone evaluator (no Sample Factory model), mirrors the PlainCNN / converted-SNN protocol\n# ----------------------------------------------------------------------------------------\nclass DirectSNN(nn.Module):\n    def __init__(self, T=8, v_init=0.5, gains=(1, 1, 1, 1), feat_scale=4.0, n_actions=4, beta=0.9):\n        super().__init__()\n        self.trunk = SpikingTrunk(3, (72, 128), T, v_init, gains, feat_scale, beta=beta)\n        self.head = nn.Linear(512, n_actions)\n\n    def forward(self, x, T=None, return_spikes=False):\n        out = self.trunk(x, T=T, return_spikes=return_spikes)\n        if return_spikes:\n            feats, spk = out\n            return self.head(feats), spk\n        return self.head(out)\n\n\ndef load_direct_snn(sf_state_dict, T, v_init=0.5, n_actions=4):\n    """Build the standalone SNN from a Sample Factory checkpoint[\'model\'] state dict."""\n    gains = sf_state_dict["encoder.trunk.gains"].cpu().tolist()\n    feat_scale = float(sf_state_dict["encoder.trunk.feat_scale"])\n    beta = float(sf_state_dict["encoder.trunk.beta"].cpu())\n    net = DirectSNN(T, v_init, gains, feat_scale, n_actions, beta=beta)\n    state = {}\n    for k, v in sf_state_dict.items():\n        if k.startswith("encoder.trunk."):\n            state["trunk." + k[len("encoder.trunk."):]] = v\n    state["head.weight"] = sf_state_dict["action_parameterization.distribution_linear.weight"]\n    state["head.bias"] = sf_state_dict["action_parameterization.distribution_linear.bias"]\n    missing, unexpected = net.load_state_dict(state, strict=False)\n    return net.eval(), missing, unexpected'
if not ENABLE_TRAINING:
    print("Plan only. Set ENABLE_TRAINING=True after reviewing cost and GPU availability.")
else:
    if os.name != "posix":raise RuntimeError("Run training on the Linux GPU server")
    SWEEP_ROOT.mkdir(parents=True,exist_ok=True)
    base_path=WORK_DIR/"sweep_lif_base.py"
    # Stable, embedded copy of our original LIF code for importable worker definitions.
    if base_path.exists() and base_path.read_text()!=MODULE_SOURCE:
        raise RuntimeError("sweep_lif_base.py differs; preserve it before starting another implementation")
    base_path.write_text(MODULE_SOURCE)
    for job in JOBS:
        name=job["name"];result_dir=SWEEP_ROOT/"results"/name
        completion=result_dir/"complete.json"
        if completion.exists():
            if json.loads(completion.read_text())!=job:raise RuntimeError("Job changed: use a new SWEEP_TAG")
            print("Reusing",name);continue
        if (SWEEP_ROOT/"train_dir"/name).exists():
            raise RuntimeError(f"Incomplete run {name}; inspect log and use a new sweep tag")
        job_path=SWEEP_ROOT/f"{name}_job.json"
        if job_path.exists() and json.loads(job_path.read_text())!=job:
            raise RuntimeError("Submitted settings changed: choose a new sweep tag")
        job_path.write_text(json.dumps(job,indent=2))
        log_path=SWEEP_ROOT/f"{name}.log"
        with log_path.open("w") as log:
            process=subprocess.Popen([sys.executable,"-u",str(WORK_DIR/"sweep_worker.py"),str(job_path)],
                cwd=WORK_DIR,stdout=log,stderr=subprocess.STDOUT,start_new_session=True)
            try:
                last_notice=0
                while process.poll() is None:
                    if time.monotonic()-last_notice>=30:
                        print("Running",name,"log:",log_path,flush=True);last_notice=time.monotonic()
                    time.sleep(2)
                if process.returncode:raise RuntimeError(f"Failed {name}. Read {log_path}")
            except BaseException:
                try:os.killpg(process.pid,signal.SIGTERM)
                except ProcessLookupError:pass
                try:process.wait(timeout=15)
                except subprocess.TimeoutExpired:
                    try:os.killpg(process.pid,signal.SIGKILL)
                    except ProcessLookupError:pass
                    process.wait()
                raise
        print("Completed",name)


## Collect the full scorecard

Load only jobs in the current plan; unrelated folders are not ranked. Partial milestone
results can be plotted but cannot win the final ranking. Inference latency is synchronized
batch-1 preprocessing + forward + action decoding after warmup; it excludes simulation
and the extra firing-rate diagnostic pass. It is not GPU power consumption.


In [ ]:
rows=[]
for job in JOBS:
    for target in MILESTONES:
        p=SWEEP_ROOT/"results"/job["name"]/f"metrics_{target}.json"
        if p.exists():
            row=json.loads(p.read_text())
            if row["config"]!=job["config"]:raise RuntimeError(f"Configuration mismatch: {p}")
            rows.append(row)
print("Loaded milestone measurements:",len(rows),"of",len(JOBS)*len(MILESTONES))
for row in rows:
    print(row["name"],"steps",row["actual_steps"],"return",round(row["mean_return"],3),
          "kills/round",row["kills_per_round"],"kills",row["mean_kills"],
          "decisions",row["mean_decisions"],"game seconds",row["mean_game_seconds"],
          "training min",row["training_seconds"]/60,"latency ms",row["latency_mean_ms"],
          "firing rates",row["firing_rates"])
if rows:
    import csv
    columns=["name","seed","target_steps","actual_steps","mean_return","std_return","kills_per_round",
             "mean_kills","mean_decisions","mean_game_seconds","training_seconds","latency_mean_ms","latency_p95_ms"]
    with (SWEEP_ROOT/"scorecard.csv").open("w",newline="") as f:
        w=csv.DictWriter(f,fieldnames=columns);w.writeheader()
        for row in rows:w.writerow({k:row[k] for k in columns})


## Charts: learning curves, efficiency, survival, compute and layer activity

Episode dispersion is saved per run. Chart error bars at the final milestone show sample standard deviation across training-run means when more than one seed is available; a single seed has no estimate of training variability.

In [ ]:
if not rows:
    print("Run experiments or select a completed sweep tag first.")
else:
    fig,axes=plt.subplots(1,2,figsize=(14,5))
    for job in JOBS:
        series=sorted([r for r in rows if r["name"]==job["name"]],key=lambda r:r["actual_steps"])
        if not series:continue
        axes[0].plot([r["actual_steps"]/1e6 for r in series],[r["mean_return"] for r in series],"o-",label=job["name"])
        axes[1].plot([r["training_seconds"]/60 for r in series],[r["mean_return"] for r in series],"o-")
    axes[0].set(xlabel="Reported training steps (millions)",ylabel="Mean original return")
    axes[1].set(xlabel="Cumulative training minutes",ylabel="Mean original return")
    axes[0].legend(fontsize=7);fig.tight_layout();plt.show()
    finals=[r for r in rows if r["target_steps"]==MILESTONES[-1]]
    if finals:
        labels=[r["name"] for r in finals]
        fig,axes=plt.subplots(2,3,figsize=(16,10))
        metrics=[("mean_return","Original return"),("kills_per_round","Kills / ammo unit"),
                 ("mean_kills","Mean kills"),("mean_game_seconds","Game seconds survived"),
                 ("training_seconds","Training seconds"),("latency_mean_ms","Batch-1 inference ms")]
        for ax,(key,title) in zip(axes.flat,metrics):
            values=[r[key] if r[key] is not None else np.nan for r in finals]
            ax.bar(labels,values);ax.set_title(title);ax.tick_params(axis="x",labelrotation=90)
        fig.tight_layout();plt.show()
        activity=np.array([r["firing_rates"] for r in finals],dtype=float)
        fig,ax=plt.subplots(figsize=(7,max(3,len(finals)*.35)))
        im=ax.imshow(activity,vmin=0,vmax=1,aspect="auto")
        ax.set_xticks(range(4),["conv1","conv2","conv3","fc"])
        ax.set_yticks(range(len(labels)),labels);ax.set_title("Firing rate: spike events / neuron-time slots")
        fig.colorbar(im,ax=ax);fig.tight_layout();plt.show()

    # Aggregate final reward by configuration, retaining seed variability.
    groups={}
    for row in finals:
        key=config_id(row["config"])
        groups.setdefault(key,[]).append(row["mean_return"])
    if groups:
        fig,ax=plt.subplots(figsize=(12,5))
        keys=list(groups)
        means=[np.mean(groups[k]) for k in keys]
        errors=[np.std(groups[k],ddof=1) if len(groups[k])>1 else 0 for k in keys]
        ax.bar(keys,means,yerr=errors,capsize=4)
        ax.set(ylabel="Mean original return",title="Final reward by configuration; error bars = sample std across training seeds")
        ax.tick_params(axis="x",labelrotation=90)
        fig.tight_layout();plt.show()


## Best completed candidate in each category

Rank by mean **original** return across all configured training seeds at the final target.
Only fully completed candidates with every planned seed qualify. Tie-breaker is higher
ammo efficiency, then lower latency. Baseline participates in each category, so the result
can correctly say that no tested modification improved it. These are validation winners,
not statistically established optima. Smoke-mode or single-seed rankings are preliminary.
Do not choose a different checkpoint rule per candidate; all use latest-at-milestone.
Use new episode seeds for final reporting, and rerun finalists across training seeds.


In [ ]:
ranked=[]
for key,item in candidates.items():
    eligible=[]
    for seed in TRAINING_SEEDS:
        name=f"lif_{key}_seed{seed}"
        final=next((r for r in rows if r["name"]==name and r["target_steps"]==MILESTONES[-1]),None)
        if final is not None and (SWEEP_ROOT/"results"/name/"complete.json").exists():eligible.append(final)
    if len(eligible)!=len(TRAINING_SEEDS):continue
    returns=np.array([r["mean_return"] for r in eligible])
    efficiencies=[r["kills_per_round"] for r in eligible]
    ranked.append(dict(id=key,config=item["config"],categories=item["categories"],
        mean_return=float(returns.mean()),across_seed_std=float(returns.std(ddof=1)) if len(returns)>1 else None,
        mean_efficiency=float(np.mean(efficiencies)) if all(v is not None for v in efficiencies) else None,
        latency_ms=float(np.mean([r["latency_mean_ms"] for r in eligible]))))
best_by_category={}
for category in CATEGORIES:
    options=[r for r in ranked if category in r["categories"]]
    expected=[key for key,item in candidates.items() if category in item["categories"]]
    if len(options)!=len(expected):
        print(category,": pending; not every candidate/seed has completed")
        continue
    winner=max(options,key=lambda r:(r["mean_return"],r["mean_efficiency"] if r["mean_efficiency"] is not None else -1,-r["latency_ms"]))
    best_by_category[category]=winner
    print(category,"BEST TESTED:",winner["config"],"mean return:",winner["mean_return"],
          "across-seed std:",winner["across_seed_std"],"kills/round:",winner["mean_efficiency"])
if best_by_category:
    (SWEEP_ROOT/"best_by_category.json").write_text(json.dumps(best_by_category,indent=2))
if SMOKE_TEST or len(TRAINING_SEEDS)<3:
    print("PRELIMINARY screening only: confirm finalists with full training and multiple seeds.")
